In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("/code/src/")

In [3]:
import json

In [4]:
from data_processing.utils.geometry_utils import get_rotation_diff, get_3d_point_distance
from visualization.viser_visualization import get_traj_frames_data
from numpy.typing import NDArray
import numpy as np

## Parameters

In [5]:
A = [ 30, 40, 7, 8, 1]
print(np.array(A)[[2, 1, 0]])

## Maincode

In [10]:
src_dir = "/data/"
dataset_root = f"{src_dir}/datasets/processed"
scene_name = "backyard_sunny"
scene_data_file =  f"{dataset_root}/{scene_name}/scene_data.json"

In [11]:
with open(scene_data_file, 'r') as f:
    scene_data = json.load(f)

In [13]:
def get_pose_weight_rot_distance(pose_a:NDArray, pose_b:NDArray, rot_w:float=0.1):
    d_t = get_3d_point_distance(pose_a[:3, 3].tolist(), pose_b[:3, 3].tolist())
    d_r = get_rotation_diff(pose_a[:3, :3], pose_b[:3,:3])

    d = np.abs(d_t) + (rot_w * np.abs(d_r))

    return d

In [14]:
def get_pose_k_nearest_neighbor(pose, ref_poses, k=1,
                                pose_distance_fn=get_pose_weight_rot_distance):
    pose_distances = []
    for ref_pose in ref_poses:
        pose_distances.append(pose_distance_fn(pose, ref_pose))
    
    nearest_ids = np.argsort(pose_distances)[:k]
    distance = sum([pose_distances[nearest_id] for nearest_id in nearest_ids]) / k
    nearest_pose = ref_poses[nearest_ids[0]]

    return nearest_pose, distance, nearest_ids

def get_pose_chamfer_distance_directed_a_to_b(traj_a_poses, 
                                             traj_b_poses, 
                                             pose_distance_fn=get_pose_weight_rot_distance,
                                             k_neighbor_size=1):
    traj_a_distances = []
    traj_a_nearest_pose = []
    
    #TODO: This is brute force need to be changed to something smarted
    for pose_a in traj_a_poses:
        nearest_pose, distance, _ = get_pose_k_nearest_neighbor(pose_a, 
                                                                traj_b_poses, 
                                                                pose_distance_fn=pose_distance_fn,
                                                                k=k_neighbor_size)
        traj_a_distances.append(distance)
        traj_a_nearest_pose.append(nearest_pose)
    
    traj_a_to_b_distance = np.mean(traj_a_distances).item()

    return traj_a_to_b_distance


def get_pose_chamfer_distance_symmetric(traj_a_poses, 
                                     traj_b_poses, 
                                     pose_distance_fn=get_pose_weight_rot_distance,
                                     k_neighbor_size=1):
    
    traj_a_to_b_distance = get_pose_chamfer_distance_directed_a_to_b(traj_a_poses=traj_a_poses,
                                                                     traj_b_poses=traj_b_poses,
                                                                     pose_distance_fn=pose_distance_fn,
                                                                     k_neighbor_size=k_neighbor_size)
    
    traj_b_to_a_distance = get_pose_chamfer_distance_directed_a_to_b(traj_a_poses=traj_b_poses,
                                                                     traj_b_poses=traj_a_poses,
                                                                     pose_distance_fn=pose_distance_fn,
                                                                     k_neighbor_size=k_neighbor_size)

    return (traj_a_to_b_distance + traj_b_to_a_distance)


def get_trajectories_diff(scene_data, traj_a_name, traj_b_name, 
                          pose_distance_fn=get_pose_weight_rot_distance, 
                          k_neighbor_size=1,
                          pose_type="colmap_pose_c2w"):
    
    traj_a = get_traj_frames_data(scene_traj_data=scene_data["trajectories"], trajectory_name=traj_a_name,
                                cam_intrinsics_type="camera_intrinsic_colmap", c2w_pose_type=pose_type)

    traj_b = get_traj_frames_data(scene_traj_data=scene_data["trajectories"], trajectory_name=traj_b_name,
                                    cam_intrinsics_type="camera_intrinsic_colmap", c2w_pose_type=pose_type)
    
    traj_a_poses = [np.array(frame["pose_c2w"]) for frame in traj_a]
    traj_b_poses = [np.array(frame["pose_c2w"]) for frame in traj_b]

    traj_mean_dis = get_pose_chamfer_distance_directed_a_to_b(traj_a_poses=traj_a_poses, 
                                                             traj_b_poses=traj_b_poses,
                                                             pose_distance_fn=pose_distance_fn,
                                                             k_neighbor_size=k_neighbor_size)
            
    return traj_mean_dis

    

In [18]:
trajectories =["orbit_inward_low", 
               "orbit_inward_mid",
               "orbit_inward_high",
               "traversal_forward_low",
               "traversal_backward_low",
               "traversal_left_low",
               "traversal_right_low",
               "orbit_inward_high",
               "panorama_360_station_a",
               "panorama_360_station_b",
               "panorama_360_station_c" ]

In [19]:
trajectories_matrix = {}
k = 1
for traj_a in trajectories:
    trajectories_matrix[traj_a] = {}
    for traj_b in trajectories:
        traj_distance = get_trajectories_diff(scene_data, 
                                              traj_a_name=traj_a, 
                                              traj_b_name=traj_b,
                                              k_neighbor_size=k)
        trajectories_matrix[traj_a][traj_b] = round(traj_distance, 3)


In [20]:
print(json.dumps(trajectories_matrix, indent=4))

In [ ]:
trajectories_matrix = {}
k = 3
for traj_a in trajectories:
    trajectories_matrix[traj_a] = {}
    for traj_b in trajectories:
        traj_distance = get_trajectories_diff(scene_data, traj_a_name=traj_a, 
                                              traj_b_name=traj_b,
                                              k_neighbor_size=k)
        trajectories_matrix[traj_a][traj_b] = round(traj_distance, 3)

In [ ]:
print(json.dumps(trajectories_matrix, indent=4))